## Use ridge resgression and cross-validation to train the unmixing model

In [ ]:
# Load packages
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn import linear_model
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import pandas as pd
import fcsparser
np.random.seed(233)
import matplotlib.pyplot as plt



In [ ]:
# read data
path = "./path/to/full-stained_raw.fcs"

In [4]:
# parse both metadata and data, get the expression values into a data frame
meta,data = fcsparser.parse(path, meta_data_only=False, reformat_meta=True)

In [5]:
# have a look
print(type(data))
data

<class 'pandas.DataFrame'>


,Time,UV1-A,UV2-A,UV3-A,UV4-A,UV5-A,UV6-A,UV7-A,UV8-A,UV9-A,...,YG9-A,YG10-A,R1-A,R2-A,R3-A,R4-A,R5-A,R6-A,R7-A,R8-A
0,0.0,542.557373,2332.536377,1800.298828,838.589966,1683.284180,3046.397705,3182.579102,2258.375000,5522.619141,...,5032.319824,3398.961670,1294.223267,4686.226074,9198.029297,7607.077637,5267.672852,4781.061523,7340.906250,5023.106445
1,1.0,654.359253,3135.995117,2978.784668,3121.844482,3437.347900,13728.028320,71248.109375,36519.480469,20043.371094,...,7403.405762,6024.258789,3837.920654,9196.717773,14498.337891,10388.107422,7938.731934,7959.092773,13394.445312,8669.881836
2,2.0,284.860138,172.830124,686.594177,1201.366211,1388.849976,4820.207520,11616.893555,8712.904297,6526.327148,...,3177.111084,3052.596924,962.631531,5719.878906,11746.294922,8685.802734,5846.497070,5523.695312,8100.785645,5892.935547
3,2.0,29.753147,570.925171,242.462555,177.516495,392.458771,449.846741,583.453491,408.464996,827.583069,...,-243.926575,-1.974022,-132.146484,-251.501328,28.929502,330.671234,-200.694809,-51.704067,-120.156120,196.599731
4,3.0,-47.255001,52.293640,-55.372818,284.909271,148.118042,1014.311157,1546.307983,1295.140015,1062.586548,...,201.698181,-365.118103,149.513916,12.451954,175.199890,264.266235,374.355865,109.599022,-38.276878,-26.612484
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1244035,2928865.0,-604.513977,1567.186035,1894.885132,2090.395996,3177.870850,5910.660156,10436.833984,6491.432129,5578.214844,...,9074.268555,5038.311035,775.931702,2112.048340,2408.204590,1820.722534,1217.205444,1148.578613,1947.660767,2072.380615
1244036,2928874.0,-83.868874,736.133545,255.318237,513.589294,868.310547,1199.615845,2643.453369,1893.337402,1461.749634,...,1772.133911,1280.836426,238.171814,301.168427,-72.182640,79.016251,-313.838654,-21.702103,-60.044773,418.414795
1244037,2928877.0,321.473999,1447.284668,1049.426147,1967.082520,2551.955078,4518.073730,10509.356445,8709.424805,7434.895996,...,1513.772827,606.024658,-423.401001,-576.377747,354.845032,-274.454987,473.090851,75.583183,138.129608,820.063721
1244038,2928879.0,393.931671,854.623474,1871.471924,2413.804688,3465.313721,6035.375977,12897.454102,11583.469727,10391.373047,...,3892.306396,3619.976318,1442.616699,5169.108398,8891.094727,6773.167480,5711.740234,5699.693359,7837.973145,6080.686523


In [6]:
# read in the spectral matrix
M = pd.read_csv("./spectral_matrix.csv")
print(type(M))

<class 'pandas.DataFrame'>


In [7]:
print(M.shape)
print(data.shape)

(64, 22)
(1244040, 71)


In [8]:
# take only the fluorescence channels from the raw data
Y = data.filter(regex="(UV|V|B|YG|R)(\d+)")

# store the other half of the data
C = data.drop(columns=Y.columns)

In [9]:
print(Y.columns)
print(Y.shape)
print(C.columns)
print(C.shape)

Index(['UV1-A', 'UV2-A', 'UV3-A', 'UV4-A', 'UV5-A', 'UV6-A', 'UV7-A', 'UV8-A',
       'UV9-A', 'UV10-A', 'UV11-A', 'UV12-A', 'UV13-A', 'UV14-A', 'UV15-A',
       'UV16-A', 'V1-A', 'V2-A', 'V3-A', 'V4-A', 'V5-A', 'V6-A', 'V7-A',
       'V8-A', 'V9-A', 'V10-A', 'V11-A', 'V12-A', 'V13-A', 'V14-A', 'V15-A',
       'V16-A', 'B1-A', 'B2-A', 'B3-A', 'B4-A', 'B5-A', 'B6-A', 'B7-A', 'B8-A',
       'B9-A', 'B10-A', 'B11-A', 'B12-A', 'B13-A', 'B14-A', 'YG1-A', 'YG2-A',
       'YG3-A', 'YG4-A', 'YG5-A', 'YG6-A', 'YG7-A', 'YG8-A', 'YG9-A', 'YG10-A',
       'R1-A', 'R2-A', 'R3-A', 'R4-A', 'R5-A', 'R6-A', 'R7-A', 'R8-A'],
      dtype='str')
(1244040, 64)
Index(['Time', 'SSC-H', 'SSC-A', 'FSC-H', 'FSC-A', 'SSC-B-H', 'SSC-B-A'], dtype='str')
(1244040, 7)


In [10]:
# transpose to have channels as row names
YT = Y.T

In [11]:
# final checks
print(YT.shape)
print(M.shape)
print(type(M))
print(type(YT))
print(M.columns)

(64, 1244040)
(64, 22)
<class 'pandas.DataFrame'>
<class 'pandas.DataFrame'>
Index(['CCR7 BV421_b', 'CD103 APC_c', 'CD127 PECy7_b', 'CD161 BUV615_b',
       'CD25 BUV737_b', 'CD27 BV786_b', 'CD28 BUV395_b', 'CD3 NFB610_b',
       'CD38_APCF810_c', 'CD39 BV480_c', 'CD4 BV650_b', 'CD45 RB780_b',
       'CD69 BB700_c', 'CD8_BUV496_c', 'CTLA4 BB515_b', 'FOXP3_PEDazzle594_c',
       'HLADQ BUV563_b', 'HLADR BV605_c', 'LD FVS440_c', 'PD1 BV711_c',
       'TIGIT PE_c', 'AF'],
      dtype='str')


In [16]:
sorted(sklearn.metrics.SCORERS.keys())

['accuracy',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'average_precision',
 'balanced_accuracy',
 'completeness_score',
 'explained_variance',
 'f1',
 'f1_macro',
 'f1_micro',
 'f1_samples',
 'f1_weighted',
 'fowlkes_mallows_score',
 'homogeneity_score',
 'jaccard',
 'jaccard_macro',
 'jaccard_micro',
 'jaccard_samples',
 'jaccard_weighted',
 'max_error',
 'mutual_info_score',
 'neg_brier_score',
 'neg_log_loss',
 'neg_mean_absolute_error',
 'neg_mean_absolute_percentage_error',
 'neg_mean_gamma_deviance',
 'neg_mean_poisson_deviance',
 'neg_mean_squared_error',
 'neg_mean_squared_log_error',
 'neg_median_absolute_error',
 'neg_root_mean_squared_error',
 'normalized_mutual_info_score',
 'precision',
 'precision_macro',
 'precision_micro',
 'precision_samples',
 'precision_weighted',
 'r2',
 'rand_score',
 'recall',
 'recall_macro',
 'recall_micro',
 'recall_samples',
 'recall_weighted',
 'roc_auc',
 'roc_auc_ovo',
 'roc_auc_ovo_weighted',
 'roc_auc_ovr',
 'roc_auc_ovr_we

In [ ]:
# Randomly selecte 75% of the channels to train the model to avoid overfitting. It perfoms better than using all data to train it from my experience.
x_train, x_test, y_train, y_test = train_test_split(M, YT, test_size=0.25, random_state=42)


In [21]:
print(y_train.shape)
print(x_train.shape)
print(x_test.shape)
print(y_test.shape)

(48, 1244040)
(48, 22)
(16, 22)
(16, 1244040)


In [22]:
# pick alpha using CV and build the best model based on smallest MSE

# Candidates generated by smallest positive one from the lg of the d from SVD of my spectra matrix. Then added some smaller values and larger ones on a log scale.
m1 = linear_model.RidgeCV(alphas=[0.1, 0.01, 0.001], scoring = "neg_mean_squared_error",store_cv_results=True) # use LOO CV by default, with negative MSE by default, use poisson deviance if you have strictly all positive values. Uses SVD solver by default when n_features < n_channels

m1.fit(x_train, y_train)
# take the best model's coefficients
print(m1.alpha_)
best_a = m1.coef_

0.01


In [ ]:
# Just a quick check
print(m1.score(x_train, y_train))
print(m1.score(x_test, y_test)) 

Best alpha: 0.01
Train R²: 0.9955450021158637
Test R²: 0.9579507254826454


In [ ]:
print(best_a.shape)

# generate a df
feature_names = M.columns

best_A = pd.DataFrame(
    m1.coef_,
    columns=feature_names
)

unmix = pd.concat([best_A, C], axis=1)

# Write to .csv
unmix.to_csv("name_unmixed.csv", index=False)

(1244040, 22)


## Visualise the alpha choices if you want

In [ ]:
print(m1.cv_results_.shape)
cv_results = m1.cv_results_ # get the all MSEs
# Aggregate MSE per channel by extracting median
med_mse_per_chl = np.median(cv_results, axis=(1))
print(mse_per_chl.shape)
print(med_mse_per_chl.shape)

# Plotting
plt.figure(figsize=(8, 5))
alphas = [0.1, 0.01, 0.001]
best_alpha = m1.alpha_
for i, sample_mse in enumerate(med_mse_per_chl):
    plt.semilogx(alphas, sample_mse, alpha=0.5, color="blue", linewidth=0.8)
# add elements
plt.axvline(best_alpha, color="red", linestyle="--", label=f"Best alpha = {best_alpha:.3f}")
plt.xlabel("alpha")
plt.ylabel("Channel median MSE")
plt.title("Median MSE vs alphas across training channels")
plt.legend()
plt.show()

(48, 1244040, 3)
